# ARENA3 PPO — 판정·확인

**P1·P2 노트북이 끝난 뒤** 새 런타임 하나에서 실행합니다.
1. 개발 판정: P1·P2와 기존 후보(G1·S2A_pre·S2A·S2B_pre·S2B)를 1단계와 같은 심판·같은 84상황으로 M4와 비교합니다. 짝 비교(P1 − G1, P2 − S2A_pre 등)도 함께 냅니다.
2. 확인: 개발 판정을 **통과한 PPO**(P1·P2 중 날짜 평균이 큰 하나)만 새 배치 C1·C2 84상황에서 다시 잽니다.
   - 기준선(M4·P0·W4)은 2단계 확인 때 돌린 것을 그대로 씁니다. 후보 252판만 돕니다(CPU 8개 기준 약 1.5시간).
   - 통과한 PPO가 없으면 SUMO를 돌리지 않고 끝납니다. 승자는 처음 한 번 고정되어, 다시 실행해도 바뀌지 않습니다.
3. P2만 돌렸다면 첫 칸의 `RUN`을 `("P2",)`로 바꾸세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3_PPO.zip"
A3 = DRIVE / "arena3"
RUN = ("P1", "P2")                     # 실제로 돌린 PPO 방식(P2만 돌렸다면 ("P2",))
JUDGE = A3 / "ppo_judge"
WORKSPACE = Path("/content/FutureTraffic_ARENA3_PPO")
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 판정 폴더:", JUDGE)
if CPUS < 8:
    print(f"주의: CPU가 {CPUS}개입니다. 확인 252판이 오래 걸립니다. 가능하면 CPU + 높은 RAM 런타임을 쓰세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_PPO_V1", "ARENA3 PPO 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PPO 설정 {package_manifest['ppo_config_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3*.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 개발 판정(84상황, M4 대비). 아직 평가가 안 끝난 PPO가 있으면 멈춥니다.
from IPython.display import display, Markdown
S2 = A3 / "stage2"
args = ["src/arena3_ppo.py", "judge", "--baseline", str(A3 / "referee" / "baseline"), "--out", str(JUDGE),
        "--reference", f"G1={A3 / 'referee' / 'G1'}"]
for v in RUN:
    args += ["--ppo", f"{v}={A3 / f'ppo_{v}' / 'eval'}"]
for n in ("S2A_pre", "S2A", "S2B_pre", "S2B"):
    args += ["--reference", f"{n}={S2 / 'eval' / n}"]
code = colab.run_live(args, WORKSPACE, check=False)
assert code == 0, "아직 평가가 끝나지 않은 PPO가 있습니다(위 '아직 평가 중'). P1·P2 노트북이 끝난 뒤 다시 실행하세요."
display(Markdown((JUDGE / "summary.md").read_text()))

In [ ]:
# 2) 확인(C1·C2 84상황): 통과한 PPO 중 날짜 평균이 가장 큰 하나만 다시 잽니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
from IPython.display import display, Markdown
CONFIRM_BASE = A3 / "stage2" / "confirm" / "baseline"   # 2단계 확인 때 돌린 M4·P0·W4(C1·C2) 252판
colab.run_live(["src/arena3_ppo.py", "confirm", "--judge", str(JUDGE), "--baseline", str(CONFIRM_BASE),
                "--out", str(JUDGE / "confirm"), "--work", "/content/ppo_confirm", "--workers", str(CPUS)], WORKSPACE)
result = colab.read_json(JUDGE / "confirm" / "confirmation.json")
print("확인 결과:", result["status"], "| 승자:", result["selection"]["winner"] if "selection" in result else None)
if (JUDGE / "confirm" / "summary.md").exists():
    display(Markdown((JUDGE / "confirm" / "summary.md").read_text()))